# Landsat 5 / TM (Thematic Mapper)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=landsat/landsat-5-tm). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1

Level 1 (Collection 2): geometrically and radiometrically calibrated digital numbers (Tier 1 / Tier 2 / RT), with metadata to convert to radiance or TOA reflectance.

Level-1 top-of-atmosphere reflectance through the Earth Engine Python API. Needs a Google account with Earth Engine access; the notebook asks for your Cloud project ID.

In [ ]:
%pip install -q earthengine-api

In [ ]:
import ee
from IPython.display import Image, display

# One-off sign-in with a Google account that has Earth Engine access
ee.Authenticate()
PROJECT = input("Google Cloud project ID registered for Earth Engine: ")
ee.Initialize(project=PROJECT)

point = ee.Geometry.Point([12.5, 41.9])           # Rome - change to your area
collection = (
    ee.ImageCollection("LANDSAT/LT05/C02/T1_TOA")
    .filterBounds(point)
    .filterDate("2005-06-01", "2005-08-31")
    .sort("CLOUD_COVER")
)
print("scenes:", collection.size().getInfo())
image = ee.Image(collection.first())
print("least cloudy scene:", image.get("system:index").getInfo(), "cloud cover:", image.get("CLOUD_COVER").getInfo())

url = image.select(['B3', 'B2', 'B1']).getThumbURL({
    "min": 0, "max": 0.3, "dimensions": 900,
    "region": point.buffer(30000).bounds(),
})
display(Image(url=url))

## L2

Level 2 (Collection 2 Science Products): atmospherically corrected surface reflectance (LEDAPS) and surface temperature (single-channel algorithm).

Searches the Microsoft Planetary Computer STAC catalogue (no account needed) and plots NDVI from the surface-reflectance bands of the least cloudy Landsat 5 scene over the period.

In [ ]:
%pip install -q pystac-client planetary-computer rioxarray matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import pystac_client
import rioxarray

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


def open_overview(href, level=2):
    """Open a COG at reduced resolution (fast), falling back to full resolution."""
    try:
        return rioxarray.open_rasterio(href, overview_level=level, masked=True).squeeze()
    except Exception:
        return rioxarray.open_rasterio(href, masked=True).squeeze()

search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=[12.4, 41.8, 12.6, 42.0],                      # Rome - change to your area
    datetime="2005-06-01/2005-08-31",
    query={"platform": {"eq": "landsat-5"}, "eo:cloud_cover": {"lt": 30}},
)
items = search.item_collection()
if not items:
    raise SystemExit("No scenes found - widen the dates or the bounding box")
item = min(items, key=lambda i: i.properties.get("eo:cloud_cover", 100))
print(len(items), "scenes found; using", item.id, item.datetime)
print("assets:", sorted(item.assets))

# Collection 2 Level-2 scale factors: reflectance = DN * 0.0000275 - 0.2
red = open_overview(item.assets["red"].href) * 0.0000275 - 0.2
nir = open_overview(item.assets["nir08"].href) * 0.0000275 - 0.2
ndvi = (nir - red) / (nir + red)
ndvi.plot(cmap="YlGn", vmin=-0.2, vmax=0.9)
plt.title("NDVI - " + item.id)
plt.show()